# 01 Naive vs Advanced RAG Architectures & LangChain Pipelines

Building end-to-end RAG pipelines using LangChain text splitters, prompt formatting, context injection, and multi-provider LLM execution.

## Setup: Repository-Level Dotenv Configuration

In [ ]:
import os
from dotenv import load_dotenv

# Load repo-level .env file
load_dotenv()

print("Phase 2 RAG Environment Loaded.")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))

## Technique 1: Document Ingestion & Chunking with RecursiveCharacterTextSplitter

In [ ]:
# LangChain Basic RAG Pipeline Setup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

# Sample Knowledge Base
raw_docs = [
    Document(page_content="PagedAttention allocates KV cache memory in non-contiguous virtual pages, eliminating 96% of VRAM fragmentation in vLLM."),
    Document(page_content="Grouped-Query Attention (GQA) groups 8 query heads per key-value head, reducing 70B model KV cache memory by 8x."),
    Document(page_content="FlashAttention tiles Query, Key, and Value matrices into fast GPU SRAM to bypass slow HBM memory bandwidth bounds.")
]

splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks = splitter.split_documents(raw_docs)

print(f"Split {len(raw_docs)} raw documents into {len(chunks)} text chunks.")
print("Sample Chunk #1:", chunks[0].page_content)

## Technique 2: Advanced Context Injection Prompt Assembly

In [ ]:
# LangChain Advanced RAG Chain with Prompt Formatting
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical AI assistant. Answer the user's question using ONLY the provided context chunks. If the answer is not in the context, say 'I do not have enough information.'"),
    ("user", "Context:\n{context}\n\nQuestion: {question}")
])

context_text = "\n\n".join([c.page_content for c in chunks[:2]])
formatted_prompt = rag_prompt.format(context=context_text, question="How does PagedAttention reduce VRAM fragmentation?")

print("Formatted Advanced RAG Prompt:\n", formatted_prompt)

## Technique 3: Multi-Provider LLM RAG Execution

In [ ]:
# Multi-Provider Execution (OpenAI, Gemini, Ollama)
print("=== Calling RAG LLM Chain ===")
try:
    from langchain_openai import ChatOpenAI
    openai_key = os.getenv("OPENAI_API_KEY")
    if openai_key:
        llm = ChatOpenAI(model="gpt-4o-mini", api_key=openai_key)
        res = llm.invoke(formatted_prompt)
        print("ChatOpenAI RAG Answer:\n", res.content)
    else:
        print("OPENAI_API_KEY not present in .env.")
except Exception as e:
    print("OpenAI RAG execution:", e)